In [1]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
SINGLE, CV = ROOT / "results", ROOT / "results_cv"

MODEL_NAMES = {"xgb": "XGBoost", "rf": "Random Forest", "et": "Extra-Trees", "svm": "SVR", "svr": "SVR",
               "knn": "kNN", "mlp": "MLP (neural network)", "enet": "ElasticNet", "pls": "PLS",
               "consensus": "Consensus (avg of top 3)", "stack": "Stacking (top 3)"}
FEATURE_SETS = {
    "C":  (13, "All 13: concentration, diameter, exposure time, core, shape, surface modification, plastic source, "
               "cell line, cell anatomical type, cell source species, cell origin, cell-tissue-organ origin, assay type"),
    "B":  (11, "C without cell line and assay type"),
    "E6": (6, "Concentration, diameter, exposure time, surface modification, core, cell line"),
    "E5": (5, "Concentration, diameter, exposure time, surface modification, core"),
    "E4": (4, "Concentration, diameter, exposure time, surface modification"),
}

In [2]:
# Single random split: one metrics.json per (feature set, model)
rows = []
for p in sorted(SINGLE.glob("features_*/*_seed42/metrics.json")):   # the original single split only
    m = json.loads(p.read_text())
    rows.append({"Evaluation": "Single random split (seed 42)", "Features": m["features"],
                 "Model": MODEL_NAMES[m["model"]], "R²": m["test_R2"], "RMSE": m["test_RMSE"], "MAE": m["test_MAE"],
                 "R² 95% CI": "—"})
single = pd.DataFrame(rows)

# Nested CV: summary written by src/cv_report.py (pooled over all rows, mean over 3 seeds)
cv_raw = pd.read_csv(CV / "summary.csv")
cv_raw = cv_raw[~cv_raw.model.str.contains("scrambled")]
cv = pd.DataFrame({"Evaluation": "Nested 5-fold CV (3 repeats)", "Features": cv_raw.features,
                   "Model": cv_raw.model.map(MODEL_NAMES), "R²": cv_raw.R2_mean, "RMSE": cv_raw.RMSE_mean,
                   "MAE": cv_raw.MAE_mean,
                   "R² 95% CI": "[" + cv_raw.R2_lo_mean.map("{:.2f}".format) + ", " + cv_raw.R2_hi_mean.map("{:.2f}".format) + "]"})

# Repeated random split: 10 independent 80/10/10 splits (seeds 0-9), written by src/repeated_report.py
rep_raw = pd.read_csv(SINGLE / "repeated_summary.csv")
rep = pd.DataFrame({"Evaluation": "Repeated random split (10 repeats)", "Features": rep_raw.Features,
                    "Model": rep_raw.Model.replace({"SVM (SVR)": "SVR"}), "R²": rep_raw["R² mean"],
                    "RMSE": rep_raw["RMSE mean"], "MAE": rep_raw["MAE mean"],
                    "R² 95% CI": "[" + rep_raw["R² 95% CI low"].map("{:.2f}".format) + ", "
                                 + rep_raw["R² 95% CI high"].map("{:.2f}".format) + "]"})

allres = pd.concat([cv, rep, single], ignore_index=True)
allres["# features"] = allres.Features.map(lambda f: FEATURE_SETS[f][0])
allres = allres[["Evaluation", "Features", "# features", "Model", "R²", "R² 95% CI", "RMSE", "MAE"]]
allres = allres.sort_values(["Evaluation", "R²"], ascending=[True, False]).reset_index(drop=True)

## Table 1: all models, all feature sets

Three evaluations: nested 5-fold CV, repeated random split (mean of 10 splits) and the single random split (seed 42). Sorted by R² within each evaluation. Green = best row of each evaluation.

In [3]:
best_idx = allres.groupby("Evaluation")["R²"].idxmax()

def highlight_best(row):
    style = "background-color: #ffff; font-weight: bold" if row.name in best_idx.values else ""
    return [style] * len(row)

(allres.style
    .apply(highlight_best, axis=1)
    .format({"R²": "{:.2f}", "RMSE": "{:.1f}", "MAE": "{:.1f}"})
    .hide(axis="index"))

Evaluation,Features,# features,Model,R²,R² 95% CI,RMSE,MAE
Nested 5-fold CV (3 repeats),C,13,Consensus (avg of top 3),0.80,"[0.72, 0.84]",10.8,6.8
Nested 5-fold CV (3 repeats),C,13,Stacking (top 3),0.79,"[0.72, 0.84]",10.9,6.9
Nested 5-fold CV (3 repeats),C,13,Extra-Trees,0.79,"[0.71, 0.83]",11.0,6.9
Nested 5-fold CV (3 repeats),C,13,XGBoost,0.77,"[0.68, 0.82]",11.5,7.5
Nested 5-fold CV (3 repeats),C,13,Random Forest,0.76,"[0.66, 0.81]",11.7,7.4
Nested 5-fold CV (3 repeats),C,13,MLP (neural network),0.72,"[0.63, 0.78]",12.5,7.9
Nested 5-fold CV (3 repeats),B,11,Stacking (top 3),0.72,"[0.63, 0.78]",12.6,8.1
Nested 5-fold CV (3 repeats),B,11,Consensus (avg of top 3),0.72,"[0.63, 0.79]",12.6,8.1
Nested 5-fold CV (3 repeats),B,11,XGBoost,0.70,"[0.61, 0.77]",13.0,8.7
Nested 5-fold CV (3 repeats),C,13,SVR,0.70,"[0.62, 0.77]",13.0,8.3


## Table 2: repeated random split (10 repeats)

Each repeat is a new stratified 80/10/10 split (seeds 0–9): tuned on the 10% validation set, scored once on the 10% test set (191 rows).
Values are **mean ± sd over the 10 repeats**. The 95% CI is for the mean (t-distribution); the range is the lowest and highest R² seen.
Green = best model.

- **Table 2a:** all 13 features, 4 models
- **Table 2b:** all feature sets (13 → 4 features), best per feature set in green

In [4]:
rep_raw = pd.read_csv(SINGLE / "repeated_summary.csv")
SETS = {"C": "All 13", "B": "11: no cell line, no assay", "E6": "6: conc., diameter, time, surface, core, cell line",
        "E5": "5: conc., diameter, time, surface, core", "E4": "4: conc., diameter, time, surface"}

def rep_table(d):
    return pd.DataFrame({
        "Features": d["Features"].map(SETS), "Model": d["Model"],
        "R² (mean ± sd)": d["R² mean"].map("{:.3f}".format) + " ± " + d["R² sd"].map("{:.3f}".format),
        "R² 95% CI": "[" + d["R² 95% CI low"].map("{:.2f}".format) + ", " + d["R² 95% CI high"].map("{:.2f}".format) + "]",
        "R² range": d["R² min"].map("{:.2f}".format) + " – " + d["R² max"].map("{:.2f}".format),
        "RMSE (mean ± sd)": d["RMSE mean"].map("{:.1f}".format) + " ± " + d["RMSE sd"].map("{:.1f}".format),
        "MAE (mean ± sd)": d["MAE mean"].map("{:.1f}".format) + " ± " + d["MAE sd"].map("{:.1f}".format),
        "_r2": d["R² mean"], "_set": d["Features"]})

GREEN = "background-color: #d1f2d9; font-weight: bold"

# Table 2a: all 13 features
t2a = rep_table(rep_raw[rep_raw.Features == "C"].sort_values("R² mean", ascending=False)).reset_index(drop=True)
best_a = t2a["_r2"].idxmax()
display(t2a.drop(columns=["_r2", "_set"]).style
        .apply(lambda r: [GREEN if r.name == best_a else ""] * len(r), axis=1).hide(axis="index")
        .set_caption("Table 2a: repeated random split, all 13 features"))

# Table 2b: every feature set, best model per set in green
order = {"C": 0, "B": 1, "E6": 2, "E5": 3, "E4": 4}
t2b = rep_table(rep_raw.assign(o=rep_raw.Features.map(order)).sort_values(["o", "R² mean"], ascending=[True, False])
                ).reset_index(drop=True)
best_b = set(t2b.groupby("_set")["_r2"].idxmax())
display(t2b.drop(columns=["_r2", "_set"]).style
        .apply(lambda r: [GREEN if r.name in best_b else ""] * len(r), axis=1).hide(axis="index")
        .set_caption("Table 2b: repeated random split, all feature sets"))

Features,Model,R² (mean ± sd),R² 95% CI,R² range,RMSE (mean ± sd),MAE (mean ± sd)
All 13,XGBoost,0.747 ± 0.100,"[0.68, 0.82]",0.55 – 0.85,12.1 ± 3.0,7.4 ± 0.9
All 13,Random Forest,0.732 ± 0.090,"[0.67, 0.80]",0.61 – 0.85,12.5 ± 2.8,7.6 ± 1.0
All 13,SVM (SVR),0.674 ± 0.102,"[0.60, 0.75]",0.47 – 0.78,13.8 ± 2.7,8.3 ± 0.9
All 13,kNN,0.630 ± 0.110,"[0.55, 0.71]",0.47 – 0.75,14.8 ± 2.9,8.7 ± 1.2


Features,Model,R² (mean ± sd),R² 95% CI,R² range,RMSE (mean ± sd),MAE (mean ± sd)
All 13,XGBoost,0.747 ± 0.100,"[0.68, 0.82]",0.55 – 0.85,12.1 ± 3.0,7.4 ± 0.9
All 13,Random Forest,0.732 ± 0.090,"[0.67, 0.80]",0.61 – 0.85,12.5 ± 2.8,7.6 ± 1.0
All 13,SVM (SVR),0.674 ± 0.102,"[0.60, 0.75]",0.47 – 0.78,13.8 ± 2.7,8.3 ± 0.9
All 13,kNN,0.630 ± 0.110,"[0.55, 0.71]",0.47 – 0.75,14.8 ± 2.9,8.7 ± 1.2
"11: no cell line, no assay",XGBoost,0.668 ± 0.077,"[0.61, 0.72]",0.56 – 0.73,14.0 ± 2.3,8.7 ± 0.9
"11: no cell line, no assay",Random Forest,0.630 ± 0.100,"[0.56, 0.70]",0.43 – 0.72,14.8 ± 2.6,9.2 ± 0.8
"11: no cell line, no assay",SVM (SVR),0.609 ± 0.132,"[0.51, 0.70]",0.29 – 0.75,15.2 ± 3.1,9.5 ± 1.3
"11: no cell line, no assay",kNN,0.514 ± 0.132,"[0.42, 0.61]",0.29 – 0.67,16.9 ± 3.2,10.2 ± 1.4
"6: conc., diameter, time, surface, core, cell line",XGBoost,0.689 ± 0.101,"[0.62, 0.76]",0.52 – 0.79,13.5 ± 2.8,8.4 ± 1.0
"6: conc., diameter, time, surface, core, cell line",Random Forest,0.661 ± 0.078,"[0.61, 0.72]",0.54 – 0.77,14.2 ± 2.4,9.0 ± 1.0


## Example: predict one experiment

Uses the **XGBoost model from the single random split** (seed 42, all 13 features, test R² 0.86). It is retrained here from its saved
best settings (`results/features_C/xgb_seed42/best_params.json`) on the same 80% training rows, so it works in any Python environment.
The example row comes from that split's **test set**, so the model never saw it during training.

- **Option 1:** change `EXAMPLE_ROW` to any test-row number (the list is printed below). You'll see the inputs, the predicted viability and the actual viability.
- **Option 2:** set `MY_INPUT` to your own experiment (edit the values). You'll get a predicted viability; there's no actual value for a new experiment.

In [5]:
import json, sys
sys.path.insert(0, "src")
from data import FEATURE_SETS as FS, load_data, random_split, xy
from models import build

df = load_data()
train, _, test = random_split(df, seed=42)                  # same split as train.py: 1,520 train / 191 test rows
X_train, y_train = xy(train, "C")
X_test, y_test = xy(test, "C")
best = json.loads(Path("results/features_C/xgb_seed42/best_params.json").read_text())
model = build("xgb", best, *FS["C"], seed=42, n_jobs=1).fit(X_train, y_train)   # trained on training rows only

# ---- Option 1: pick a test row -------------------------------------------------------------
EXAMPLE_ROW = 1266          # PS, 57 nm, amino-modified, 50 ug/mL, 72 h, Caco-2 cells

# ---- Option 2: your own experiment (set MY_INPUT = None to use Option 1) -------------------
MY_INPUT = None
# MY_INPUT = {"Concentration(μg/mL)": 100, "Diameter(nm)": 50, "Exposure time": 24,
#             "Core": "PS", "Shape": "Beads", "Surface modification": "Amino-modified", "Plastic source": "Commercial",
#             "Cell source species": "Human", "Cell anatomical type": "Epithelial", "Cell-tissue-organ-origin": "Lung",
#             "Cell origin": "Cell-line", "Cell line": "A549", "Assay type": "MTT"}

if MY_INPUT is None:
    x = X_test.loc[[EXAMPLE_ROW]]
    actual = test.loc[EXAMPLE_ROW, "Cell viability"]
else:
    x = pd.DataFrame([MY_INPUT])[X_test.columns]
    actual = None

predicted = float(model.predict(x)[0])
display(x.T.rename(columns={x.index[0]: "Input"}))
print(f"Predicted viability: {predicted:.1f}%")
if actual is not None:
    print(f"Actual viability:    {actual:.1f}%")
    print(f"Error:               {predicted - actual:+.1f} percentage points")
print(f"\nTest-row numbers you can use for EXAMPLE_ROW: {sorted(X_test.index)[:15]} ... ({len(X_test)} rows)")

Loaded model_data_with_ref.csv: 1902 rows, 1 duplicate dropped -> 1901 rows, 71 papers


,Input
Concentration(μg/mL),50.0
Diameter(nm),56.93
Exposure time,72
Core,PS
Shape,Beads
Surface modification,Amino-modified
Plastic source,Commercial
Cell source species,Human
Cell anatomical type,Epithelial
Cell-tissue-organ-origin,Colon


Predicted viability: 34.0%
Actual viability:    36.1%
Error:               -2.1 percentage points

Test-row numbers you can use for EXAMPLE_ROW: [2, 5, 11, 19, 20, 32, 35, 44, 70, 76, 101, 117, 126, 133, 137] ... (191 rows)
